In [4]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
import shutil
from sklearn.metrics import accuracy_score

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

model_name = "MNIST_MLP_HGQ_StaticTraining_model"

In [5]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_test = X_test.astype("float32") / 255
y_test = keras.utils.to_categorical(y_test, 10)

In [6]:
from keras.models import load_model
import hgq.layers

model = load_model(f"{model_name}.keras")
score = model.evaluate(X_test, y_test)

 10/313 ━━━━━━━━━━━━━━━━━━━━ 25s 83ms/step - accuracy: 0.9688 - loss: 0.1720

I0000 00:00:1777541689.172045  251504 service.cc:152] XLA service 0x7ea664003420 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777541689.172082  251504 service.cc:160]   StreamExecutor device (0): Host, Default Version
2026-04-30 11:34:49.198758: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777541689.348967  251504 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9570 - loss: 0.4473


In [7]:

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense (QDense)                │ (None, 32)             │       100,485 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_1 (QDense)              │ (None, 32)             │         4,229 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_2 (QDense)              │ (None, 10)             │         1,325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 265,089 (957.84 KB)

 Trainable params: 79,524 (310.64 KB)

 Non-trainable params: 26,515 (25.91 KB)

 Optimizer params: 159,050 (621.29 KB)

In [ ]:
import hls4ml

output_dir = f"hls4ml_prj_{model_name}"

hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

#hls_config['Model']['Strategy'] = 'Distributed Arithmetic'
#proj_name = f"{str(model_to_test)}_{str(model_revision)}_hls4ml_prj_{str(hls4ml_revision)}"

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='vitis',
    hls_config=hls_config,
    #io_type='io_stream',
    #proj_name = proj_name,
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
)
hls_model.compile()

In [9]:
hls_model.build(csim=False, synth=True, cosim=False)


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Thu Apr 30 11:35:05 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
Sourcing Tcl script '/home/ncgadmin/DAT255/DAT255-project/MNIST/MLP_HGQ_StaticTraining/hls4ml_prj_{model_name}/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html
INFO: [HLS 200-10] Creating and opening solution '/home/ncgadmin/DAT255/DAT255-project/MNIST/MLP_HGQ_StaticTraining/hls4ml_prj_{model_name}/myproject_prj'.
INFO: [HLS 200-1510] Running: set_top myproject 
INFO: [HLS 200-1510] Running: add_files firmware/myproject.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' 

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '3.648',
  'BestLatency': '7',
  'WorstLatency': '7',
  'IntervalMin': '1',
  'IntervalMax': '1',
  'DSP': '0',
  'FF': '14443',
  'LUT': '219809',
  'BRAM_18K': '0',
  'URAM': '0',
  'AvailableBRAM_18K': '288',
  'AvailableDSP': '1248',
  'AvailableFF': '234240',
  'AvailableLUT': '117120',
  'AvailableURAM': '64'}}